In [1]:
from gplearn.genetic import SymbolicTransformer
from gplearn.functions import make_function
from sklearn.utils.random import check_random_state

import numpy as np
import array
import pandas
import matplotlib
import math   # 导入 math 模块
import matplotlib.pyplot as plt
import pandas as pd
from pandas import Series, DataFrame
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from itertools import combinations
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
OL= LinearRegression(normalize=False)

plt.rcParams['savefig.dpi'] = 100 #图片像素
plt.rcParams['figure.dpi'] = 100#分辨率

In [2]:
data_train=pandas.read_csv(r"C:\Users\z\Desktop\图表及相应源代码\Fig 4\F_train.csv")
data_test=pandas.read_csv(r"C:\Users\z\Desktop\图表及相应源代码\Fig 4\F_test.csv")

In [3]:
F_train=np.array(data_train)
p_train=data_train.ev_ratio
p_train=np.array(p_train)
F_0train=F_train[:,3:67]

In [4]:
F_test=np.array(data_test)
p_test=data_test.ev_ratio
p_test=np.array(p_test)
F_0test=F_test[:,3:67]

In [5]:
crossover=[]
parsimony=[]
for i in range(25,85,2):
    crossover.append(i*0.01)

for i in range(5,55,10):
    parsimony.append(i*0.00002)
    
crossover_2=[]
subtree_mutation=[]
hoist_mutation=[]

for k in range(len(crossover)):
    up=math.floor((1-crossover[k])*100/3)
    low=math.floor((0.92-crossover[k])*100/3)
    for i in range(low,up,15):
        subtree_mutation.append(i*0.01)
        hoist_mutation.append(i*0.01)
        crossover_2.append(crossover[k])
        
point_mutation=[]
point_mutation=np.array(1-np.array(crossover_2)-np.array(subtree_mutation)-np.array(hoist_mutation))
tot=np.c_[crossover_2,subtree_mutation,hoist_mutation,point_mutation]

In [6]:
rs_0 = ShuffleSplit(n_splits=5, test_size=.80, random_state=0)#
rs_1 = ShuffleSplit(n_splits=5, test_size=.75, random_state=0)#
rs_2 = ShuffleSplit(n_splits=5, test_size=.70, random_state=0)#
rs_3 = ShuffleSplit(n_splits=5, test_size=.65, random_state=0)#
rs_4 = ShuffleSplit(n_splits=5, test_size=.60, random_state=0)#
rs_5 = ShuffleSplit(n_splits=5, test_size=.55, random_state=0)#
rs_6 = ShuffleSplit(n_splits=5, test_size=.50, random_state=0)#
rs_7 = ShuffleSplit(n_splits=5, test_size=.45, random_state=0)#
rs_8 = ShuffleSplit(n_splits=5, test_size=.40, random_state=0)#
rs_9 = ShuffleSplit(n_splits=5, test_size=.35, random_state=0)#
rs_10= ShuffleSplit(n_splits=5, test_size=.30, random_state=0)#
rs_11= ShuffleSplit(n_splits=5, test_size=.25, random_state=0)#
rs_12= ShuffleSplit(n_splits=5, test_size=.20, random_state=0)#
rs_13= ShuffleSplit(n_splits=5, test_size=.15, random_state=0)#
rs_14= ShuffleSplit(n_splits=5, test_size=.10, random_state=0)#

tr=[]
te=[]

In [7]:
for train_index, test_index in rs_0.split(F_0train):
    tr.append(train_index)
    te.append(test_index)

In [22]:
function_set = [ 'add','sub','mul', 'div', 'sqrt', 'log','abs','neg', 'inv','sin','cos','tan']

n=4

F_tr=F_0train[tr[n],:]

p_tr=p_train[tr[n]]
#p_te=p[te[n]]

j=len(parsimony)

gen=35
#len(tot)
for m in range(0,j):
    for i in range(len(tot)):
        
        gp = SymbolicTransformer(population_size=10500,tournament_size=20,
                                 metric='pearson',function_set = function_set,init_depth=(4, 15),
                                 generations=gen, stopping_criteria=0.9,
                                 hall_of_fame=500, n_components=1,
                                 p_crossover=tot[i][0],p_subtree_mutation=tot[i][1],
                                 p_hoist_mutation=tot[i][2],p_point_mutation=tot[i][3],
                                 max_samples=0.9, verbose=1,n_jobs=3,
                                 parsimony_coefficient=parsimony[m], random_state=0)

        gp.fit(F_tr,(p_tr.reshape(-1,1)-1)*100)
        
        for program in gp:
            print(program)
            print(program.length_)
            print(program.raw_fitness_)
        
        gp_features = gp.transform(F_tr)
        OL.fit(gp_features, (p_tr.reshape(-1,1)-1)*100)
        p_trpre=OL.predict(gp_features)
        r_2tr=r2_score((p_tr.reshape(-1,1)-1)*100, p_trpre)
        print(r_2tr)
        
        gp_features_te = gp.transform(F_0test)
        OL.fit(gp_features_te, (p_te.reshape(-1,1)-1)*100)
        p_tepre=OL.predict(gp_features_te)
        r_2te=r2_score((p_te.reshape(-1,1)-1)*100, p_test)
        print(r_2te)
    
        gp_features_0 = gp.transform(F_0)
        OL.fit(gp_features_0, (p.reshape(-1,1)-1)*100)
        p_pre=OL.predict(gp_features_0)
        r_2=r2_score((p.reshape(-1,1)-1)*100, p_pre)
        #result=pd.DataFrame(np.c_[(p.reshape(-1,1)-1)*100,p_pre])
        print(r_2)

D:\Python\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    49.76        0.0882536       13          0.56685         0.547794      7.39m
   1     8.14         0.260382       12          0.59913         0.313226      2.87m
   2     4.39          0.39245        2         0.599867         0.177414      3.05m
   3     4.23         0.428053        3         0.612073         0.218398      3.33m
   4     4.25         0.434659        6         0.624677         0.114722      3.08m
   5     4.54         0.436756       14         0.630852         0.369669      3.36m
   6     6.07         0.429999        5         0.658952         0.233387      3.01m
   7     7.11          0.43234        8         0.653882         0.412319      3.09m
   8     7.84          0.43987        8         0.662456         0.228067  

D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its de

   0    49.76        0.0882536       13          0.56685         0.547794      4.17m
   1     8.08         0.262124       12          0.59913         0.313226      3.45m
   2     4.35         0.396198        4          0.60263         0.290404      3.88m
   3     4.22         0.433331        3         0.612073         0.218398      2.75m
   4     4.21         0.439338        6         0.624677         0.114722      5.31m
   5     4.53         0.440932       14         0.630852         0.369669      4.52m
   6     6.02         0.433861        7         0.647889         0.444386      4.41m
   7     7.01          0.43955        8         0.653882         0.412319      4.10m
   8     7.42         0.450356        7         0.662997         0.294027      3.85m
   9     7.96         0.446484        7         0.676233         0.220164      3.69m
  10     8.51         0.436897       10         0.674012         0.276468      3.70m
  11     8.68          0.42817        8         0.678905        0

D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its de

...

D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its de

   0    49.76        0.0882536       13          0.56685         0.547794      3.71m
   1     5.30         0.282479       31         0.613508         0.742144      2.76m
   2     3.49         0.456514       27         0.624543         0.635923      2.73m
   3     3.59         0.513564       27         0.664717         0.145429      2.60m
   4     4.88         0.514247       27         0.656942         0.347632      2.57m
   5     8.31         0.507724       27         0.671701         0.296251      2.62m
   6     8.08          0.51793       29         0.671303         0.184069      2.59m
   7     7.47         0.530277       10         0.667262         0.272611      2.37m
   8     7.38           0.5322        7         0.670425         0.185023      2.32m
   9     7.56         0.530768        8         0.676929         0.161216      2.27m
  10     7.67          0.53368        8         0.671843         0.348068      2.14m
  11     7.81         0.532262        8         0.674311         

D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its de

   0    49.76        0.0882536       13          0.56685         0.547794      3.58m
   1     5.26         0.284625       31         0.613508         0.742144      2.75m
   2     3.43         0.461374       27         0.624543         0.635923      2.68m
   3     3.55         0.517669       27         0.664717         0.145429      2.61m
   4     4.81         0.517258       27         0.656942         0.347632      2.67m
   5     7.90         0.509376       27         0.671701         0.296251      2.58m
   6     7.77          0.51383        6         0.691643         0.243785      2.52m
   7     6.99         0.512181        6         0.683815         0.370565      2.47m
   8     6.60         0.501865       10         0.689197         0.349866      2.34m
   9     6.52          0.49609       13         0.690389         0.124584      2.18m
  10     6.68         0.502074        6         0.703709         0.179321      2.14m
  11     6.80         0.502246       13         0.691387         

D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its de

In [ ]:
mul(X13, log(div(div(log(sqrt(div(tan(add(X40, sub(X40, X6))), div(X34, X13)))), X50), abs(X59))))